In [ ]:
# Suppress TensorFlow info/warning logs to keep output clean
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Standard libraries for numerical ops, plotting, and evaluation metrics
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, 
    classification_report, 
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    roc_curve,
    auc,
    roc_auc_score
)
from sklearn.preprocessing import label_binarize
from tensorflow.keras.models import load_model
from utils.load_data import load_dataset
from utils.preprocessing import split_data, create_data_generators
import time

# Make sure the results directory exists before saving any output files
os.makedirs("results", exist_ok=True)

# Collect ground-truth labels and model predictions across all batches
def get_predictions(model, generator):
    generator.reset()
    
    y_true = []
    y_pred = []
    y_pred_proba = []
    
    for i in range(len(generator)):
        X_batch, y_batch = generator[i]
        pred_batch = model.predict(X_batch, verbose=0)
        y_true.extend(y_batch)
        y_pred.extend(np.argmax(pred_batch, axis=1))
        y_pred_proba.extend(pred_batch)
    
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_pred_proba = np.array(y_pred_proba)
    
    return y_true, y_pred, y_pred_proba

# Plot and save the confusion matrix as a heatmap for a given model
def plot_confusion_matrix(y_true, y_pred, class_names, model_name, filename):
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(14, 12))
    sns.heatmap(
        cm, 
        annot=True, 
        fmt='d', 
        cmap='Blues',
        xticklabels=class_names,
        yticklabels=class_names,
        cbar_kws={'label': 'Number of Samples'},
        linewidths=0.5,
        linecolor='gray'
    )
    
    plt.title(f'Confusion Matrix - {model_name}', fontsize=14, pad=15)
    plt.xlabel('Predicted Label', fontsize=11)
    plt.ylabel('True Label', fontsize=11)
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.yticks(rotation=0, fontsize=8)
    
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: {filename}")

# Plot ROC curves for each class (one-vs-rest) plus micro and macro averages
def plot_roc_curves(y_true, y_pred_proba, class_names, model_name, filename):
    n_classes = len(class_names)
    
    y_true_bin = label_binarize(y_true, classes=range(n_classes))
    
    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_pred_proba[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    
    # Compute micro-average ROC curve across all classes
    fpr["micro"], tpr["micro"], _ = roc_curve(y_true_bin.ravel(), y_pred_proba.ravel())
    roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])
    
    # Compute macro-average ROC curve by interpolating across all FPR points
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= n_classes
    fpr["macro"] = all_fpr
    tpr["macro"] = mean_tpr
    roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])
    
    plt.figure(figsize=(12, 10))
    
    plt.plot(fpr["micro"], tpr["micro"],
             label=f'Micro-average (AUC = {roc_auc["micro"]:.3f})',
             color='deeppink', linestyle=':', linewidth=3)
    
    plt.plot(fpr["macro"], tpr["macro"],
             label=f'Macro-average (AUC = {roc_auc["macro"]:.3f})',
             color='navy', linestyle=':', linewidth=3)
    
    for i in range(n_classes):
        plt.plot(fpr[i], tpr[i], alpha=0.3, linewidth=1)
    
    plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=11)
    plt.ylabel('True Positive Rate', fontsize=11)
    plt.title(f'ROC Curves (One-vs-Rest) - {model_name}', fontsize=14)
    plt.legend(loc='lower right', fontsize=9)
    plt.grid(True, alpha=0.3, linestyle='--')
    
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: {filename}")
    
    return roc_auc["macro"]

# Side-by-side training curves for Scratch CNN and Transfer Learning
def plot_training_curves_comparison(scratch_history_path, transfer_history_path):
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 12))
    
    try:
        scratch_history = np.load(scratch_history_path, allow_pickle=True).item()
        
        epochs_s = range(1, len(scratch_history['accuracy']) + 1)
        
        ax1.plot(epochs_s, scratch_history['accuracy'], 'b-', label='Training', linewidth=2)
        ax1.plot(epochs_s, scratch_history['val_accuracy'], 'b--', label='Validation', linewidth=2)
        ax1.set_title('Scratch CNN - Accuracy', fontsize=12, fontweight='bold')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Accuracy')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        ax3.plot(epochs_s, scratch_history['loss'], 'b-', label='Training', linewidth=2)
        ax3.plot(epochs_s, scratch_history['val_loss'], 'b--', label='Validation', linewidth=2)
        ax3.set_title('Scratch CNN - Loss', fontsize=12, fontweight='bold')
        ax3.set_xlabel('Epoch')
        ax3.set_ylabel('Loss')
        ax3.legend()
        ax3.grid(True, alpha=0.3)
        
        print("Loaded Scratch CNN training history")
    except:
        ax1.text(0.5, 0.5, 'Training history not available\nRun train.py first', 
                ha='center', va='center', transform=ax1.transAxes, fontsize=11)
        ax3.text(0.5, 0.5, 'Training history not available\nRun train.py first', 
                ha='center', va='center', transform=ax3.transAxes, fontsize=11)
        print("Scratch CNN training history not found")
    
    try:
        transfer_history = np.load(transfer_history_path, allow_pickle=True).item()
        
        epochs_t = range(1, len(transfer_history['accuracy']) + 1)
        
        ax2.plot(epochs_t, transfer_history['accuracy'], 'r-', label='Training', linewidth=2)
        ax2.plot(epochs_t, transfer_history['val_accuracy'], 'r--', label='Validation', linewidth=2)
        ax2.set_title('Transfer Learning - Accuracy', fontsize=12, fontweight='bold')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Accuracy')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        ax4.plot(epochs_t, transfer_history['loss'], 'r-', label='Training', linewidth=2)
        ax4.plot(epochs_t, transfer_history['val_loss'], 'r--', label='Validation', linewidth=2)
        ax4.set_title('Transfer Learning - Loss', fontsize=12, fontweight='bold')
        ax4.set_xlabel('Epoch')
        ax4.set_ylabel('Loss')
        ax4.legend()
        ax4.grid(True, alpha=0.3)
        
        print("Loaded Transfer Learning training history")
    except:
        ax2.text(0.5, 0.5, 'Training history not available\nRun train_transfer_learning.py first', 
                ha='center', va='center', transform=ax2.transAxes, fontsize=11)
        ax4.text(0.5, 0.5, 'Training history not available\nRun train_transfer_learning.py first', 
                ha='center', va='center', transform=ax4.transAxes, fontsize=11)
        print("Transfer Learning training history not found")
    
    plt.tight_layout()
    plt.savefig('results/training_curves_comparison.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("Saved: results/training_curves_comparison.png")

# Return a dict of common classification metrics for easy comparison
def calculate_metrics(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision_macro': precision_score(y_true, y_pred, average='macro'),
        'recall_macro': recall_score(y_true, y_pred, average='macro'),
        'f1_macro': f1_score(y_true, y_pred, average='macro'),
        'precision_weighted': precision_score(y_true, y_pred, average='weighted'),
        'recall_weighted': recall_score(y_true, y_pred, average='weighted'),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted'),
    }

# Run full evaluation for a single model: metrics, confusion matrix, ROC curve
def evaluate_model(model, test_generator, class_names, model_name):
    print(f"\n{'='*60}")
    print(f"Evaluating: {model_name}")
    print(f"{'='*60}")
    
    y_true, y_pred, y_pred_proba = get_predictions(model, test_generator)
    metrics = calculate_metrics(y_true, y_pred)
    
    print(f"  Accuracy:  {metrics['accuracy']:.4f}")
    print(f"  Precision: {metrics['precision_weighted']:.4f}")
    print(f"  Recall:    {metrics['recall_weighted']:.4f}")
    print(f"  F1-Score:  {metrics['f1_weighted']:.4f}")
    
    plot_confusion_matrix(
        y_true, y_pred, class_names, model_name,
        f'results/confusion_matrix_{model_name.lower().replace(" ", "_")}.png'
    )
    
    roc_auc = plot_roc_curves(
        y_true, y_pred_proba, class_names, model_name,
        f'results/roc_curve_{model_name.lower().replace(" ", "_")}.png'
    )
    metrics['roc_auc'] = roc_auc
    print(f"  ROC AUC:   {roc_auc:.4f} (macro-average)")
    
    # Identify the best and worst classified classes from the confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    per_class_accuracy = cm.diagonal() / cm.sum(axis=1)
    best_idx = np.argmax(per_class_accuracy)
    worst_idx = np.argmin(per_class_accuracy)
    
    print(f"  Best classified: '{class_names[best_idx]}' ({per_class_accuracy[best_idx]:.2%})")
    print(f"  Worst classified: '{class_names[worst_idx]}' ({per_class_accuracy[worst_idx]:.2%})")
    
    return y_true, y_pred, y_pred_proba, metrics

# Dashboard with 4 subplots: metrics comparison, training time, ROC AUC, and improvement
def create_comparison_chart(scratch_metrics, transfer_metrics, scratch_time, transfer_time):
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 12))
    
    metrics_names = ['Accuracy', 'Precision\n(weighted)', 'Recall\n(weighted)', 'F1-Score\n(weighted)']
    scratch_values = [
        scratch_metrics['accuracy'],
        scratch_metrics['precision_weighted'],
        scratch_metrics['recall_weighted'],
        scratch_metrics['f1_weighted']
    ]
    transfer_values = [
        transfer_metrics['accuracy'],
        transfer_metrics['precision_weighted'],
        transfer_metrics['recall_weighted'],
        transfer_metrics['f1_weighted']
    ]
    
    x = np.arange(len(metrics_names))
    width = 0.35
    
    bars1 = ax1.bar(x - width/2, scratch_values, width, label='Scratch CNN', 
                    color='#3498db', alpha=0.85, edgecolor='black', linewidth=1)
    bars2 = ax1.bar(x + width/2, transfer_values, width, label='Transfer Learning', 
                    color='#e74c3c', alpha=0.85, edgecolor='black', linewidth=1)
    
    ax1.set_ylabel('Score', fontsize=11, fontweight='bold')
    ax1.set_title('Performance Metrics Comparison', fontsize=13, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(metrics_names, fontsize=9)
    ax1.legend(fontsize=10, loc='lower right')
    ax1.grid(True, alpha=0.3, axis='y', linestyle='--')
    ax1.set_ylim([0, 1.0])
    
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                   f'{height:.3f}', ha='center', va='bottom', fontsize=8)
    
    times = [scratch_time, transfer_time]
    bars = ax2.bar(['Scratch CNN', 'Transfer\nLearning'], times, 
                   color=['#3498db', '#e74c3c'], alpha=0.85, edgecolor='black', linewidth=1)
    ax2.set_ylabel('Training Time (seconds)', fontsize=11, fontweight='bold')
    ax2.set_title('Training Time Comparison', fontsize=13, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='y', linestyle='--')
    
    for bar, time_val in zip(bars, times):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + max(times)*0.02,
               f'{time_val:.1f}s\n({time_val/60:.1f} min)', 
               ha='center', va='bottom', fontsize=9)
    
    if 'roc_auc' in scratch_metrics and 'roc_auc' in transfer_metrics:
        roc_values = [scratch_metrics['roc_auc'], transfer_metrics['roc_auc']]
        bars = ax3.bar(['Scratch CNN', 'Transfer\nLearning'], roc_values,
                       color=['#3498db', '#e74c3c'], alpha=0.85, edgecolor='black', linewidth=1)
        ax3.set_ylabel('ROC AUC (macro)', fontsize=11, fontweight='bold')
        ax3.set_title('ROC AUC Comparison', fontsize=13, fontweight='bold')
        ax3.grid(True, alpha=0.3, axis='y', linestyle='--')
        ax3.set_ylim([0, 1.0])
        
        for bar, val in zip(bars, roc_values):
            height = bar.get_height()
            ax3.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                   f'{val:.3f}', ha='center', va='bottom', fontsize=9)
    
    # Calculate relative improvement of transfer learning over scratch CNN per metric
    improvement = {}
    for i, metric in enumerate(['Accuracy', 'Precision', 'Recall', 'F1-Score']):
        if scratch_values[i] > 0:
            improvement[metric] = ((transfer_values[i] - scratch_values[i]) / scratch_values[i]) * 100
        else:
            improvement[metric] = 0
    
    colors = ['#27ae60' if v > 0 else '#e74c3c' for v in improvement.values()]
    bars = ax4.bar(improvement.keys(), improvement.values(), color=colors, 
                   alpha=0.85, edgecolor='black', linewidth=1)
    ax4.set_ylabel('Improvement (%)', fontsize=11, fontweight='bold')
    ax4.set_title('Transfer Learning Improvement Over Scratch CNN', fontsize=13, fontweight='bold')
    ax4.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax4.grid(True, alpha=0.3, axis='y', linestyle='--')
    
    for bar, val in zip(bars, improvement.values()):
        height = bar.get_height()
        y_pos = height + 0.5 if height > 0 else height - 2
        ax4.text(bar.get_x() + bar.get_width()/2., y_pos,
               f'{val:+.1f}%', ha='center', va='bottom' if height > 0 else 'top', 
               fontsize=9, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('results/model_comparison_dashboard.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("Saved: results/model_comparison_dashboard.png")

# Bar chart comparing per-class accuracy between the two models
def create_per_class_comparison(y_true_s, y_pred_s, y_true_t, y_pred_t, class_names):
    cm_s = confusion_matrix(y_true_s, y_pred_s)
    cm_t = confusion_matrix(y_true_t, y_pred_t)
    
    per_class_s = cm_s.diagonal() / cm_s.sum(axis=1)
    per_class_t = cm_t.diagonal() / cm_t.sum(axis=1)
    
    fig, ax = plt.subplots(figsize=(16, 8))
    
    x = np.arange(len(class_names))
    width = 0.35
    
    ax.bar(x - width/2, per_class_s, width, label='Scratch CNN', 
           color='#3498db', alpha=0.85, edgecolor='black', linewidth=0.5)
    ax.bar(x + width/2, per_class_t, width, label='Transfer Learning', 
           color='#e74c3c', alpha=0.85, edgecolor='black', linewidth=0.5)
    
    ax.set_xlabel('Class', fontsize=11, fontweight='bold')
    ax.set_ylabel('Accuracy', fontsize=11, fontweight='bold')
    ax.set_title('Per-Class Accuracy Comparison', fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=8)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3, axis='y', linestyle='--')
    ax.set_ylim([0, 1.0])
    
    plt.tight_layout()
    plt.savefig('results/per_class_comparison.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("Saved: results/per_class_comparison.png")

# Print a structured summary table with winners highlighted per metric
def print_comparison_table(scratch_metrics, transfer_metrics, scratch_time, transfer_time, 
                          scratch_params, transfer_params):
    print(f"\n{'='*80}")
    print("MODEL COMPARISON TABLE")
    print(f"{'='*80}")
    
    print(f"\n{'Metric':<30} {'Scratch CNN':<20} {'Transfer Learning':<20}")
    print(f"{'-'*70}")
    
    print(f"{'Training Accuracy':<30} {scratch_metrics.get('train_acc', 'N/A'):<20} "
          f"{transfer_metrics.get('train_acc', 'N/A'):<20}")
    
    print(f"{'Validation Accuracy':<30} {scratch_metrics.get('val_acc', 'N/A'):<20} "
          f"{transfer_metrics.get('val_acc', 'N/A'):<20}")
    
    print(f"{'Training Time (seconds)':<30} {scratch_time:<20.1f} {transfer_time:<20.1f}")
    
    print(f"{'Trainable Parameters':<30} {scratch_params:<20,} {transfer_params:<20,}")
    
    print(f"\n{'-'*70}")
    print("TEST SET PERFORMANCE")
    print(f"{'-'*70}")
    
    winner_acc = "Scratch" if scratch_metrics['accuracy'] > transfer_metrics['accuracy'] else "Transfer"
    if abs(scratch_metrics['accuracy'] - transfer_metrics['accuracy']) < 0.001:
        winner_acc = "Tie"
    print(f"{'Accuracy':<30} {scratch_metrics['accuracy']:<20.4f} "
          f"{transfer_metrics['accuracy']:<20.4f} ({winner_acc})")
    
    winner_prec = "Scratch" if scratch_metrics['precision_weighted'] > transfer_metrics['precision_weighted'] else "Transfer"
    if abs(scratch_metrics['precision_weighted'] - transfer_metrics['precision_weighted']) < 0.001:
        winner_prec = "Tie"
    print(f"{'Precision (weighted)':<30} {scratch_metrics['precision_weighted']:<20.4f} "
          f"{transfer_metrics['precision_weighted']:<20.4f} ({winner_prec})")
    
    winner_rec = "Scratch" if scratch_metrics['recall_weighted'] > transfer_metrics['recall_weighted'] else "Transfer"
    if abs(scratch_metrics['recall_weighted'] - transfer_metrics['recall_weighted']) < 0.001:
        winner_rec = "Tie"
    print(f"{'Recall (weighted)':<30} {scratch_metrics['recall_weighted']:<20.4f} "
          f"{transfer_metrics['recall_weighted']:<20.4f} ({winner_rec})")
    
    winner_f1 = "Scratch" if scratch_metrics['f1_weighted'] > transfer_metrics['f1_weighted'] else "Transfer"
    if abs(scratch_metrics['f1_weighted'] - transfer_metrics['f1_weighted']) < 0.001:
        winner_f1 = "Tie"
    print(f"{'F1-Score (weighted)':<30} {scratch_metrics['f1_weighted']:<20.4f} "
          f"{transfer_metrics['f1_weighted']:<20.4f} ({winner_f1})")
    
    if 'roc_auc' in scratch_metrics and 'roc_auc' in transfer_metrics:
        print(f"{'ROC AUC (macro)':<30} {scratch_metrics['roc_auc']:<20.4f} "
              f"{transfer_metrics['roc_auc']:<20.4f}")
    
    print(f"{'='*80}")

# ── Main execution ─────────────────────────────────────────────────────────
print("="*60)
print("Model Evaluation and Comparison")
print("="*60)

# Load dataset and extract class names from the label map
print("\nLoading dataset...")
X, y, label_map = load_dataset("Dataset")
class_names = list(label_map.keys())
print(f"Loaded {len(X)} images across {len(class_names)} classes")

# Split into train/val/test and build data generators for each model type
print("\nSplitting dataset...")
X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y)

print("\nCreating test generators...")
_, _, test_gen_scratch = create_data_generators(
    X_train, y_train, X_val, y_val, X_test, y_test
)

# Transfer learning expects 3-channel input, so replicate grayscale channel
X_train_rgb = np.repeat(X_train, 3, axis=-1)
X_val_rgb = np.repeat(X_val, 3, axis=-1)
X_test_rgb = np.repeat(X_test, 3, axis=-1)

_, _, test_gen_transfer = create_data_generators(
    X_train_rgb, y_train, X_val_rgb, y_val, X_test_rgb, y_test
)

# Load Scratch CNN, evaluate it, and retrieve training time if available
print("\nLoading Scratch CNN model...")
try:
    scratch_model = load_model('models/scratch_cnn_best.keras')
    print("Scratch CNN model loaded")
    
    y_true_s, y_pred_s, y_pred_proba_s, scratch_metrics = evaluate_model(
        scratch_model, test_gen_scratch, class_names, "Scratch CNN"
    )
    scratch_params = scratch_model.count_params()
    
    scratch_time = 0
    try:
        with open('models/scratch_training_time.txt', 'r') as f:
            scratch_time = float(f.read())
    except:
        scratch_time = 0
        print("  Note: Training time not available")
        
except Exception as e:
    print(f"Error loading Scratch CNN: {e}")
    print("Run train.py first")
    exit()

# Load Transfer Learning model, evaluate it, and retrieve training time
print("\nLoading Transfer Learning model...")
try:
    transfer_model = load_model('models/transfer_learning_best.keras')
    print("Transfer Learning model loaded")
    
    y_true_t, y_pred_t, y_pred_proba_t, transfer_metrics = evaluate_model(
        transfer_model, test_gen_transfer, class_names, "Transfer Learning"
    )
    transfer_params = transfer_model.count_params()
    
    transfer_time = 0
    try:
        with open('models/transfer_training_time.txt', 'r') as f:
            transfer_time = float(f.read())
    except:
        transfer_time = 0
        print("  Note: Training time not available")
        
except Exception as e:
    print(f"Error loading Transfer Learning model: {e}")
    print("Run train_transfer_learning.py first")
    exit()

# Generate and save all comparison visualizations
print("\nPlotting training curves...")
plot_training_curves_comparison(
    'models/scratch_training_history.npy',
    'models/transfer_training_history.npy'
)

print("\nCreating comparison visualizations...")
create_comparison_chart(
    scratch_metrics, transfer_metrics, 
    scratch_time, transfer_time
)

create_per_class_comparison(
    y_true_s, y_pred_s, y_true_t, y_pred_t, class_names
)

print_comparison_table(
    scratch_metrics, transfer_metrics,
    scratch_time, transfer_time,
    scratch_params, transfer_params
)

# Write a plain-text evaluation report summarizing metrics and classification details
print("\nSaving evaluation report...")
with open('results/evaluation_report.txt', 'w', encoding='utf-8') as f:
    f.write("="*80 + "\n")
    f.write("EVALUATION REPORT\n")
    f.write("Arabic Handwritten Character Classification\n")
    f.write("="*80 + "\n\n")
    
    f.write("SCRATCH CNN PERFORMANCE\n")
    f.write("-"*40 + "\n")
    f.write(f"Accuracy:  {scratch_metrics['accuracy']:.4f}\n")
    f.write(f"Precision: {scratch_metrics['precision_weighted']:.4f}\n")
    f.write(f"Recall:    {scratch_metrics['recall_weighted']:.4f}\n")
    f.write(f"F1-Score:  {scratch_metrics['f1_weighted']:.4f}\n")
    f.write(f"ROC AUC:   {scratch_metrics.get('roc_auc', 'N/A'):.4f}\n")
    f.write(f"Parameters: {scratch_params:,}\n\n")
    
    f.write("TRANSFER LEARNING PERFORMANCE\n")
    f.write("-"*40 + "\n")
    f.write(f"Accuracy:  {transfer_metrics['accuracy']:.4f}\n")
    f.write(f"Precision: {transfer_metrics['precision_weighted']:.4f}\n")
    f.write(f"Recall:    {transfer_metrics['recall_weighted']:.4f}\n")
    f.write(f"F1-Score:  {transfer_metrics['f1_weighted']:.4f}\n")
    f.write(f"ROC AUC:   {transfer_metrics.get('roc_auc', 'N/A'):.4f}\n")
    f.write(f"Parameters: {transfer_params:,}\n\n")
    
    f.write("CLASSIFICATION REPORTS\n")
    f.write("="*40 + "\n\n")
    
    f.write("Scratch CNN:\n")
    f.write(classification_report(y_true_s, y_pred_s, target_names=class_names))
    f.write("\n\nTransfer Learning:\n")
    f.write(classification_report(y_true_t, y_pred_t, target_names=class_names))

print("Saved: results/evaluation_report.txt")

print(f"\n{'='*60}")
print("Evaluation complete")
print(f"{'='*60}")
